# Phase 2: Noise Profiling and Data Exploration (AeroSonicDB `gt_train.csv`)

This notebook implements Phase 2 using `dataset/AeroSonicDB/gt_train.csv` instead of `manifest.csv`.

It builds a manifest-like table, computes session-level noise profiles from no-aircraft segments, and generates the required exploration plots.

## 1) Set Up Notebook Environment and Paths

In [15]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf

# Make local project modules importable when running from notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.noise_profiling import NoiseProfiler
from keras_yamnet.preprocessing import preprocess_input

plt.style.use('seaborn-v0_8-whitegrid')

GT_CSV = PROJECT_ROOT / 'dataset' / 'AeroSonicDB' / 'gt_train.csv'
AUDIO_ROOT = Path('C:\\Users\\imborhau\\OneDrive - NTNU\Documents\\Prosjektoppgave\\AeroSonicDB-YPAD0523\data\\raw\\audio\\1')  # update this if WAV files are elsewhere
NOISE_PROFILE_DIR = PROJECT_ROOT / 'outputs' / 'noise_profiles'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'exploration'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEATHER_MAP = {
    'session1': 'Clear',
    'session2': 'Wind',
    'session3': 'Rain',
    'session4': 'Snow',
    'session5': 'Wind+Rain',
}

print(f'Project root: {PROJECT_ROOT.resolve()}')
print(f'GT CSV: {GT_CSV.resolve()}')
print(f'Audio root: {AUDIO_ROOT}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')

Project root: C:\Users\imborhau\Documents\sound-event-detection-aircrafts
GT CSV: C:\Users\imborhau\Documents\sound-event-detection-aircrafts\dataset\AeroSonicDB\gt_train.csv
Audio root: C:\Users\imborhau\OneDrive - NTNU\Documents\Prosjektoppgave\AeroSonicDB-YPAD0523\data\raw\audio\1
Output dir: C:\Users\imborhau\Documents\sound-event-detection-aircrafts\outputs\exploration


## 2) Load `gt_train.csv` and Inspect Available Columns

In [11]:
df_gt = pd.read_csv(GT_CSV, sep=None, engine='python')
print(df_gt.dtypes)
df_gt.head()

filename      object
start_time     int64
end_time       int64
class          int64
location       int64
fold           int64
dtype: object


,filename,start_time,end_time,class,location,fold
0,000000_2022-12-08_09-05-46_0_0.wav,0,10,0,0,1
1,000000_2022-12-08_09-06-16_0_0.wav,0,10,0,0,1
2,000000_2022-12-08_09-06-46_0_0.wav,0,10,0,0,1
3,000000_2022-12-08_09-07-16_0_0.wav,0,10,0,0,1
4,000000_2022-12-08_09-07-46_0_0.wav,0,10,0,0,1


## 3) Create a Manifest-Like DataFrame from `gt_train.csv`

We convert `gt_train.csv` fields into a table with the core columns expected by Phase 2 logic.

In [12]:
def _session_from_fold(fold_value):
    if pd.isna(fold_value):
        return 'unknown'
    text = str(fold_value)
    return f'session{int(float(text))}' if text.replace('.', '', 1).isdigit() else text

manifest_like = pd.DataFrame({
    'filename': df_gt['filename'].astype(str),
    'start_time': df_gt['start_time'].astype(float),
    'end_time': df_gt['end_time'].astype(float),
    'class': df_gt['class'].astype(int),
    'location': df_gt['location'].astype(str),
    'session': df_gt['fold'].map(_session_from_fold) if 'fold' in df_gt.columns else 'session1',
    'dataset': 'AeroSonicDB',
})

manifest_like.head()

,filename,start_time,end_time,class,location,session,dataset
0,000000_2022-12-08_09-05-46_0_0.wav,0.0,10.0,0,0,session1,AeroSonicDB
1,000000_2022-12-08_09-06-16_0_0.wav,0.0,10.0,0,0,session1,AeroSonicDB
2,000000_2022-12-08_09-06-46_0_0.wav,0.0,10.0,0,0,session1,AeroSonicDB
3,000000_2022-12-08_09-07-16_0_0.wav,0.0,10.0,0,0,session1,AeroSonicDB
4,000000_2022-12-08_09-07-46_0_0.wav,0.0,10.0,0,0,session1,AeroSonicDB


## 4) Map/Derive Required Fields (`session`, `location`, `label`, `npy_path`)

In [13]:
def build_audio_index(audio_root: Path) -> dict[str, Path]:
    return {path.name: path for path in audio_root.rglob('*.wav')}


def resolve_audio_path(filename: str, audio_index: dict[str, Path]) -> Path:
    if Path(filename).exists():
        return Path(filename)
    if filename in audio_index:
        return audio_index[filename]
    raise FileNotFoundError(f'Could not resolve {filename}. Update AUDIO_ROOT.')


def load_segment_wav(row: pd.Series, audio_index: dict[str, Path]):
    wav_path = resolve_audio_path(row['filename'], audio_index)
    info = sf.info(str(wav_path))
    start_frame = max(0, int(float(row['start_time']) * info.samplerate))
    end_frame = max(start_frame + 1, int(float(row['end_time']) * info.samplerate))
    wav, sr = sf.read(str(wav_path), start=start_frame, stop=end_frame, dtype='int16')
    if wav.ndim > 1:
        wav = wav.mean(axis=1).astype(np.int16)
    return wav, sr


def segment_to_mel_db(wav: np.ndarray, sr: int) -> np.ndarray:
    _, mel_spec = preprocess_input(wav, sr)
    # preprocess_input returns time x mel; convert to mel x time
    if mel_spec.shape[0] > mel_spec.shape[1]:
        mel_spec = mel_spec.T
    return mel_spec


def rms_from_db_spectrogram(spec_db: np.ndarray) -> float:
    return float(np.sqrt(np.mean(10 ** (spec_db / 10))))


def load_noise_profiles(profile_dir: Path) -> dict[str, dict[str, np.ndarray]]:
    profiles = {}
    for file_path in sorted(profile_dir.glob('*_noise_profile.npz')):
        session_id = file_path.stem.replace('_noise_profile', '')
        data = np.load(file_path)
        profiles[session_id] = {
            'mean': data['mean'],
            'std': data['std'],
            'median': data['median'],
            'p5': data['p5'],
            'p95': data['p95'],
            'n_segments': int(data['n_segments']),
        }
    return profiles


manifest_like['label'] = manifest_like['class'].map({0: 'no_aircraft', 1: 'aircraft'})
manifest_like['npy_path'] = np.nan  # optional; kept for schema compatibility
required_cols = ['filename', 'start_time', 'end_time', 'session', 'location', 'label', 'class']
manifest_like = manifest_like.dropna(subset=['filename', 'start_time', 'end_time'])
missing_cols = [c for c in required_cols if c not in manifest_like.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

manifest_like[required_cols].head()

,filename,start_time,end_time,session,location,label,class
0,000000_2022-12-08_09-05-46_0_0.wav,0.0,10.0,session1,0,no_aircraft,0
1,000000_2022-12-08_09-06-16_0_0.wav,0.0,10.0,session1,0,no_aircraft,0
2,000000_2022-12-08_09-06-46_0_0.wav,0.0,10.0,session1,0,no_aircraft,0
3,000000_2022-12-08_09-07-16_0_0.wav,0.0,10.0,session1,0,no_aircraft,0
4,000000_2022-12-08_09-07-46_0_0.wav,0.0,10.0,session1,0,no_aircraft,0


## 5) Implement and Run `NoiseProfiler.compute_session_profile`

In [16]:
audio_index = build_audio_index(AUDIO_ROOT)
print(f'Indexed WAV files: {len(audio_index)}')

profiler = NoiseProfiler(
    audio_root=AUDIO_ROOT,
    session_column='session',
    label_column='class',
    filename_column='filename',
    start_time_column='start_time',
    end_time_column='end_time',
    no_aircraft_value=0,
)

session_ids = sorted(manifest_like['session'].unique())
print('Sessions:', session_ids)

if session_ids:
    sample_session = session_ids[0]
    sample_profile = profiler.compute_session_profile(manifest_like, sample_session, extractor=None)
    print(f"Sample profile computed for {sample_session}: n_segments={sample_profile['n_segments']}")
else:
    print('No sessions found in manifest_like.')

Indexed WAV files: 625
Sessions: ['session1', 'session2', 'session3', 'session4', 'session5']


FileNotFoundError: Could not resolve audio file '000000_2022-12-08_09-05-46_0_0.wav'. Set audio_root to the folder containing WAV files.

## 6) Compute All Session Profiles and Save `.npz` + Summary JSON

In [ ]:
profiles = profiler.compute_all_profiles(manifest_like, extractor=None)
NOISE_PROFILE_DIR.mkdir(parents=True, exist_ok=True)
profiler.save_profiles(profiles, NOISE_PROFILE_DIR)

summary_path = NOISE_PROFILE_DIR / 'noise_profile_summary.json'
print('Saved summary:', summary_path)
if summary_path.exists():
    print(json.loads(summary_path.read_text()) )

profiles = load_noise_profiles(NOISE_PROFILE_DIR)
print('Loaded profiles:', sorted(profiles.keys()))

## 7) Generate Per-Session Aircraft Example Plots (Waveform + Spectrogram)

In [ ]:
df_aircraft = manifest_like[manifest_like['class'] == 1].copy()
example_rows = []
for session_id, group in df_aircraft.groupby('session'):
    group_sorted = group.sort_values('start_time').reset_index(drop=True)
    example_rows.append(group_sorted.iloc[len(group_sorted) // 2])

if example_rows:
    fig, axes = plt.subplots(len(example_rows), 2, figsize=(14, max(10, 3 * len(example_rows))))
    if len(example_rows) == 1:
        axes = np.array([axes])

    for row_idx, row in enumerate(example_rows):
        wav, sr = load_segment_wav(row, audio_index)
        spec_db = segment_to_mel_db(wav, sr)

        t = np.arange(len(wav)) / sr + float(row['start_time'])
        axes[row_idx, 0].plot(t, wav)
        axes[row_idx, 0].set_title(f"{row['session']} ({WEATHER_MAP.get(row['session'], row['session'])}) - Waveform")
        axes[row_idx, 0].set_xlabel('Time (s)')
        axes[row_idx, 0].set_ylabel('Amplitude')

        librosa.display.specshow(
            spec_db,
            x_axis='time',
            y_axis='mel',
            sr=sr,
            hop_length=160,
            fmax=7500,
            ax=axes[row_idx, 1],
            cmap='magma',
        )
        axes[row_idx, 1].set_title(f"{row['session']} - {row['filename']}")

    plt.tight_layout()
    out_path = OUTPUT_DIR / 'per_session_examples.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved:', out_path)
else:
    print('No aircraft rows found in manifest_like.')

## 8) Generate Noise Profile Overlay Plot (Mean ± Std)

In [ ]:
if profiles:
    plt.figure(figsize=(12, 6))
    for session_id, profile in sorted(profiles.items()):
        mean = profile['mean']
        std = profile['std']
        x = np.arange(len(mean))
        label = f"{session_id} ({WEATHER_MAP.get(session_id, session_id)})"
        plt.plot(x, mean, label=label)
        plt.fill_between(x, mean - std, mean + std, alpha=0.15)

    plt.xlabel('Mel band index')
    plt.ylabel('Mean power (dB)')
    plt.title('Average noise spectrum per session (mean ± std)')
    plt.legend(loc='best')
    plt.tight_layout()
    out_path = OUTPUT_DIR / 'noise_profile_overlay.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved:', out_path)
else:
    print('No profiles loaded. Run Section 6 first.')

## 9) Generate Class Balance and Session × Location Count Table

In [ ]:
class_counts = (
    manifest_like
    .groupby(['session', 'label'])
    .size()
    .reset_index(name='count')
)

plt.figure(figsize=(12, 6))
sns.barplot(data=class_counts, x='session', y='count', hue='label')
plt.title('Class balance per session')
plt.xlabel('Session')
plt.ylabel('Segment count')
plt.tight_layout()
class_balance_path = OUTPUT_DIR / 'class_balance_by_session.png'
plt.savefig(class_balance_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', class_balance_path)

pivot_counts = (
    manifest_like
    .pivot_table(index='session', columns='location', values='filename', aggfunc='count', fill_value=0)
    .sort_index()
)
pivot_counts['Total'] = pivot_counts.sum(axis=1)
pivot_counts.loc['Total'] = pivot_counts.sum(axis=0)
display(pivot_counts)

fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')
table = ax.table(
    cellText=pivot_counts.values,
    rowLabels=pivot_counts.index,
    colLabels=pivot_counts.columns,
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.4)
table_path = OUTPUT_DIR / 'segment_count_table.png'
plt.savefig(table_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', table_path)

## 10) Compute RMS Energy Distributions and Save All Figures

In [ ]:
rows_for_rms = manifest_like.copy()
if len(rows_for_rms) > 1500:
    rows_for_rms = rows_for_rms.sample(1500, random_state=42)

rms_records = []
for _, row in rows_for_rms.iterrows():
    try:
        wav, sr = load_segment_wav(row, audio_index)
        spec_db = segment_to_mel_db(wav, sr)
        rms_records.append({
            'session': row['session'],
            'label': row['label'],
            'rms': rms_from_db_spectrogram(spec_db),
        })
    except FileNotFoundError:
        continue

rms_df = pd.DataFrame(rms_records)
if not rms_df.empty:
    plt.figure(figsize=(12, 6))
    try:
        sns.kdeplot(
            data=rms_df,
            x='rms',
            hue='session',
            style='label',
            common_norm=False,
            fill=False,
        )
    except Exception:
        sns.histplot(
            data=rms_df,
            x='rms',
            hue='session',
            element='step',
            stat='density',
            common_norm=False,
            bins=40,
        )
    plt.title('RMS energy distribution by session and class')
    plt.xlabel('RMS energy')
    plt.ylabel('Density')
    plt.tight_layout()
    rms_path = OUTPUT_DIR / 'rms_energy_distributions.png'
    plt.savefig(rms_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved:', rms_path)
else:
    print('No RMS rows computed. Check AUDIO_ROOT and file availability.')

print('\nGenerated files:')
for p in sorted(OUTPUT_DIR.glob('*.png')):
    print('-', p.name)